# Levanter (fixed) — Colab training + HF export

This notebook sets up a **working** Levanter training run on Colab (GPU), following the approach that worked in your second snippet.

**What’s fixed vs the broken version:**
- Installs Levanter with deps (so `tqdm_loggable` et al. are present)
- Pins `equinox<0.13,!=0.12.0`
- Aligns `jax==0.7.2`, `jaxlib==0.7.2`, and `jax-cuda12-plugin==0.7.2`
- Forces `protobuf<6` to avoid Colab conflicts while keeping Levanter running
- Uses `--trainer.tracker '{type: noop}'` and disables JAXPR/HLO logs
- GPT‑2 *sampling* is done via **HF export** (Levanter’s `sample_lm.py` targets LLaMA)

You can also optionally train **LLaMA-small** and sample with Levanter’s built‑in script.

In [1]:
# 0) Mount Google Drive (for persistent repo/cache/checkpoints)
try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
except Exception as e:
  print("Not in Colab or drive mount failed; continuing without Drive.")

import os, pathlib
PROJECT_ROOT = "/content/drive/MyDrive/colab_projects/levanter"
pathlib.Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
%cd $PROJECT_ROOT

# Cache root for datasets/tokenizers/checkpoints
CACHE_ROOT = os.path.join(PROJECT_ROOT, "cache")
pathlib.Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["CACHE_ROOT"] = CACHE_ROOT
print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("Using CACHE_ROOT:", CACHE_ROOT)

Mounted at /content/drive
/content/drive/MyDrive/colab_projects/levanter
Using PROJECT_ROOT: /content/drive/MyDrive/colab_projects/levanter
Using CACHE_ROOT: /content/drive/MyDrive/colab_projects/levanter/cache


## 1) Clone / sync the repo (your branch)
If your feature branch isn’t available, change `BRANCH` to `main`.

In [2]:
import os, subprocess, sys
REPO_URL = "https://github.com/chris544460/levanter.git"
BRANCH = "feat/style-prefix-token"  # set to 'main' if needed

def run(cmd):
  print("$", cmd)
  status = subprocess.call(cmd, shell=True)
  if status != 0:
    raise SystemExit(f"Command failed: {cmd}")

if not os.path.exists(os.path.join(PROJECT_ROOT, ".git")):
  run(f"git clone -b {BRANCH} --single-branch {REPO_URL} {PROJECT_ROOT}")
else:
  run(f"git -C {PROJECT_ROOT} fetch origin {BRANCH}")
  run(f"git -C {PROJECT_ROOT} checkout {BRANCH} || true")
  run(f"git -C {PROJECT_ROOT} reset --hard origin/{BRANCH}")
  run(f"git -C {PROJECT_ROOT} clean -fd")

!git -C "$PROJECT_ROOT" rev-parse --abbrev-ref HEAD
!git -C "$PROJECT_ROOT" log -1 --oneline

$ git -C /content/drive/MyDrive/colab_projects/levanter fetch origin feat/style-prefix-token
$ git -C /content/drive/MyDrive/colab_projects/levanter checkout feat/style-prefix-token || true
$ git -C /content/drive/MyDrive/colab_projects/levanter reset --hard origin/feat/style-prefix-token
$ git -C /content/drive/MyDrive/colab_projects/levanter clean -fd
feat/style-prefix-token
8bd5aeb1 (HEAD -> feat/style-prefix-token, origin/feat/style-prefix-token) docs/notebooks: add training pipeline figures; update train_from_readme; update draccus __init__


## 2) Install Levanter (editable) + **compatibility pins**
We:
1. Install Levanter in editable mode **with** its dependencies (fixes missing modules like `tqdm_loggable`).
2. Pin `equinox<0.13,!=0.12.0` (Levanter still uses older APIs).
3. Force `protobuf<6` to keep Colab’s other libs happy.
4. Install `jax[cuda12]==0.7.2` to match `jaxlib` and the CUDA12 PJRT plugin.

In [3]:
%cd $PROJECT_ROOT
!python -m pip install -U pip wheel setuptools

# Install Levanter with dependencies (editable)
!python -m pip install -e .

# Pin Equinox (>=0.11.x, but <0.13 and not 0.12.0) to avoid API breakage
!python -m pip install "equinox<0.13,!=0.12.0"

# Force protobuf to <6 to avoid conflicts with Colab preinstalls
!python -m pip install --force-reinstall --no-deps "protobuf>=5.26.1,<6"

# Ensure JAX + CUDA12 plugin match (prevents PJRT aborts)
!python -m pip install -U "jax[cuda12]==0.7.2"

/content/drive/MyDrive/colab_projects/levanter
Obtaining file:///content/drive/MyDrive/colab_projects/levanter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached protobuf-6.32.1-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)


## 3) Verify imports and devices
If you still see a device mismatch warning, restart the runtime and re‑run from the top.

In [4]:
cd ..

/content/drive/MyDrive/colab_projects


In [5]:
import jax, levanter, importlib, os
print("JAX version:", jax.__version__)
try:
  import jax_cuda12_plugin as _jcp
  print("jax-cuda12-plugin:", getattr(_jcp, "__version__", "present"))
except Exception as e:
  print("jax-cuda12-plugin not importable (OK on CPU runtimes)")
print("JAX devices:", jax.devices())
print("Levanter version:", getattr(levanter, "__version__", "dev"))

/content/drive/MyDrive/colab_projects/levanter/src/levanter/optim/model_averaging.py:53: SyntaxWarning: invalid escape sequence \"\s\"
  """Hybrid EMA followed by :math:`1 - \\sqrt{x}` decay.
/content/drive/MyDrive/colab_projects/levanter/src/levanter/optim/model_averaging.py:103: SyntaxWarning: invalid escape sequence \"\s\"
  """EMA followed by :math:`1 - \\sqrt{x}` decay."""


JAX version: 0.7.2
jax-cuda12-plugin: present
JAX devices: [CudaDevice(id=0)]
Levanter version: 1.2


## 4) Train GPT‑2 “nano” (toy demo)
This follows the README quickstart. It runs for ~100 steps on WikiText‑103. We disable W&B and heavy debug logs.

In [6]:
%cd $PROJECT_ROOT
import os
from draccus import parse as draccus_parse
from levanter.main.train_lm import TrainLmConfig, main as train_lm_main
from levanter.tracker.tracker import NoopConfig

# Parse the YAML directly and override settings programmatically to avoid CLI parsing issues
cfg = draccus_parse(TrainLmConfig, "config/gpt2_nano.yaml", args=[])
cfg.trainer.tracker = NoopConfig()
cfg.trainer.log_jaxprs = False
cfg.trainer.log_xla_hlo = False

train_lm_main(cfg)

/content/drive/MyDrive/colab_projects/levanter


## 5) Export the latest checkpoint to Hugging Face format (for GPT‑2) and sample
Levanter’s built‑in sampler targets LLaMA; for GPT‑2 we export to HF and use `transformers` to generate.

In [ ]:
import os, glob, json
%cd $PROJECT_ROOT
cands = [p for p in glob.glob(os.path.join('checkpoints', '*', 'step-*'))
         if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
  try:
    with open(os.path.join(p, 'metadata.json')) as f:
      m = json.load(f)
    return (m.get('timestamp', ''), int(m.get('step', -1)))
  except Exception:
    return ('', -1)
if not cands:
  raise SystemExit('No checkpoints found under checkpoints/. Run the training cell above first.')
LATEST = sorted(cands, key=score)[-1]
print('Latest checkpoint =>', LATEST)

In [ ]:
%cd $PROJECT_ROOT
!python -m levanter.main.export_lm_to_hf \
  --checkpoint_path "$LATEST" \
  --output_dir /tmp/gpt2_nano_hf \
  --model.type gpt2 \
  --model.hidden_dim 32 \
  --model.num_layers 2 \
  --model.num_heads 4

In [ ]:
# Sample with transformers (CPU is fine for this tiny model)
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("/tmp/gpt2_nano_hf")
model = AutoModelForCausalLM.from_pretrained("/tmp/gpt2_nano_hf")
prompt = "Question: What's the capital of Germany?\nAnswer:"
inputs = tok(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=32)
print(tok.decode(outputs[0], skip_special_tokens=True))
print("\n(Expect random-ish text; this tiny toy model doesn’t learn facts.)")

## 6) (Optional) Train LLaMA‑small on OpenWebText
This one builds caches into Drive and trains a small LLaMA config. You can later sample with Levanter’s `sample_lm.py`.

> If HF requires auth for your tokenizer, set `huggingface-cli login` in a separate cell.

In [ ]:
%cd $PROJECT_ROOT
import os
from draccus import parse as draccus_parse
from levanter.main.train_lm import TrainLmConfig, main as train_lm_main
from levanter.tracker.tracker import NoopConfig

# Parse the YAML and set cache dir + tracker overrides programmatically
cfg = draccus_parse(TrainLmConfig, "config/llama_small_fast.yaml", args=[])
cfg.data.cache_dir = os.path.join(os.environ.get("CACHE_ROOT", "."), "openwebtext")
cfg.trainer.tracker = NoopConfig()
cfg.trainer.log_jaxprs = False
cfg.trainer.log_xla_hlo = False

train_lm_main(cfg)

### Sample the most recent LLaMA‑small checkpoint (uses Levanter’s sampler)
Update the model dims if you edited the config.

In [ ]:
%cd $PROJECT_ROOT
import os, glob, json
cands = [p for p in glob.glob(os.path.join('checkpoints', '*', 'step-*'))
         if os.path.exists(os.path.join(p, 'metadata.json'))]
def score(p):
  try:
    with open(os.path.join(p, 'metadata.json')) as f:
      m = json.load(f)
    return (m.get('timestamp', ''), int(m.get('step', -1)))
  except Exception:
    return ('', -1)
if not cands:
  raise SystemExit('No checkpoints found. Train LLaMA-small first.')
LATEST_LLAMA = sorted(cands, key=score)[-1]
print('Latest checkpoint =>', LATEST_LLAMA)

!python -m levanter.main.sample_lm \
  --checkpoint_path "$LATEST_LLAMA" \
  --tokenizer NousResearch/Llama-2-7b-hf \
  --model.type llama \
  --model.hidden_dim 768 \
  --model.intermediate_dim 2048 \
  --model.num_heads 12 \
  --model.num_kv_heads 12 \
  --model.num_layers 12 \
  --model.seq_len 1024 \
  --temperature 0.0 \
  --max_new_tokens 16 \
  --prompts "What is the capital of France?"

---
## (Optional) Use an isolated **uv** environment
If you want Levanter to keep `protobuf>=6` while Colab stays on `<6`, you can run via `uv` (it creates a `.venv` under the repo):

In [ ]:
%cd $PROJECT_ROOT
!python -m pip install -U uv

# Ensure Equinox pin inside the uv venv and correct JAX GPU stack
!uv run pip install "equinox<0.13,!=0.12.0" "jax[cuda12]==0.7.2"

In [ ]:
# Train GPT-2 nano under uv env (same flags)
%cd $PROJECT_ROOT
!uv run --extra gpu python -m levanter.main.train_lm \
  --config_path config/gpt2_nano.yaml \
  --trainer.tracker '{type: noop}' \
  --trainer.log_jaxprs false \
  --trainer.log_xla_hlo false

## Notes & gotchas
- **GPU plugin mismatch**: If you see PJRT errors mentioning `abort_collectives_on_failure` or “plugin version not compatible”, re‑run the JAX install cell (`jax[cuda12]==0.7.2`).
- **Equinox API**: Keep `equinox<0.13,!=0.12.0` unless the repo updates its calls.
- **Protobuf**: On “plain pip” path we force `<6` because Colab ships libs that require it. If you want to adhere to Levanter’s `protobuf>=6`, use the **uv** path.
- **Sampling GPT‑2**: Use the HF export route (Levanter’s `sample_lm.py` checks for LLaMA).
- **W&B**: The training cells use `{type: noop}`. If you want W&B, run `wandb login` and drop the tracker override.
- **Persistence**: Checkpoints and caches live under Drive (`PROJECT_ROOT`) so you don’t lose them when the Colab VM resets.